In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

csv_path = os.path.join(path, "Q1_data.csv")

# Reading the dataset
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(8, 6))
sns.histplot(df['Delivery_Time'], kde=True)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.show()

# I decided to use a histogram with a KDE (that smooth line) because I need to see the shape of the data.
# If it looks like a bell curve, my regression model will likely work well.

In [ ]:
# Task 1: Write your code here:
# dropping the Order_ID column
df = df.drop(columns=['Order_ID'])
# I'm dropping 'Order_ID' because it's just a unique label for each row.
# It doesn't actually affect how long the delivery takes (it's not a feature),
# and if I leave it in, the model might try to find a weird pattern in the ID numbers, which leads to overfitting.

In [ ]:
# Task 2: Write your code here:
# checking which columns have missing values
print(df.isnull().sum())

In [ ]:
# Task 3: Write your code here:
# First, I have to deal with the missing Target values (Delivery_Time).
# I see 106 missing values in Delivery_Time.
# I decided to DROP these rows entirely.
# Because Delivery_Time is the answer I'm trying to predict.
# If I fill it with an average, I'm basically training my model on fake answers, which is cheating.
df = df.dropna(subset=['Delivery_Time'])

# Now for the other columns:
# 'Courier_Experience_yrs' is a number, so I'll fill it with the Median.
# I chose median because if one courier has 50 years experience (an outlier), the mean would get skewed.
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())

# Weather, Traffic_Level, and Time_of_Day are text (categorical).
# I can't calculate an average for "Rainy", so I'm filling them with the Mode (the most common value).
cols_to_fill = ['Weather', 'Traffic_Level', 'Time_of_Day']
for col in cols_to_fill:
    df[col] = df[col].fillna(df[col].mode()[0])

# Checking to make sure all zeros are gone
print(df.isnull().sum())

In [ ]:
# checking for duplicates first
print(f"Duplicates before: {df.duplicated().sum()}")

# removing them
df = df.drop_duplicates()

# Checking after
print(f"Duplicates after: {df.duplicated().sum()}")

# I'm removing duplicates because if the exact same order data appears twice,
# the model will over-prioritize that specific scenario, leading to bias.

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)

df = pd.get_dummies(df, columns=['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'], drop_first=True)

# displaying the new columns to verify
df.head()

In [ ]:
def one_hot_encode(y, num_classes):
    y = np.array(y)
    m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    for i in range(m):
        # Identify which class this sample belongs to
        class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        one_hot[i, class_label] = 1

    return one_hot

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

# I need to separate the Features (X) from the Target (y) before scaling.
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

scaler = StandardScaler()

# I'm fitting the scaler on X and transforming it.
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Checking the result
X_scaled.head()

# I did this because Distance_km has big numbers (like 10-20) but Courier_Experience_yrs has small numbers (like 1-5).
# If I don't scale them to the same range, the model will think Distance is more important just because the number is bigger.

In [ ]:
# Task 6: Write your code here:
# Task: Check for target imbalance

# Since this is a regression problem, we don't have classes to count.
# Instead, I am checking the "Skewness" of the delivery times.
# If the data is heavily skewed (leaning to the left or right), it acts like an imbalance.

skewness = df['Delivery_Time'].skew()
print(f"Skewness value: {skewness}")

# Plotting it to visualize the balance
plt.figure(figsize=(6, 4))
sns.histplot(df['Delivery_Time'], kde=True)
plt.title('Distribution of Delivery Time')
plt.show()

# My Conclusion:
# If the skewness is near 0 (e.g., between -0.5 and 0.5), it is NOT imbalanced (it's normal).
# If the number is high (like > 1.0), it IS imbalanced (highly skewed).

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# 1. Split the dataset into features (X) and target (y)
# I'm separating them because the model needs to know what to learn from (X) and what to predict (y).
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Step 1: Defining the Split
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# Step 2: Defining the Model
model = RandomForestRegressor(random_state=42)

mae_scores = []

# Step 3: Training Loop

# I'm looping through 5 different splits of the data to make sure my model isn't just lucky on one specific test set.
for train_index, test_index in kf.split(X_scaled):

    # Splitting the data
    X_train, X_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Training
    model.fit(X_train, y_train)

    # Predicting
    y_pred = model.predict(X_test)

    # Calculating Error
    # I used MAE (Mean Absolute Error) because it's easy to interpret.
    # It tells me exactly how many minutes my prediction is off by on average.
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

# Step 4: Final Output

# I'm averaging the scores to get one final performance number.
average_mae = np.mean(mae_scores)

print(f"MAE Scores for each fold: {mae_scores}")
print(f"Average MAE: {average_mae}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Plot 1: Feature Importance ---

plt.figure(figsize=(10, 6))

# I'm extracting the importance scores from the random forest model.
# I matched them with the column names so I know which score belongs to which feature.
feat_importances = pd.Series(model.feature_importances_, index=X_scaled.columns)

# I'm plotting only the top 10 features.
# I did this because if I plot all of them, the chart gets messy and hard to read.
feat_importances.nlargest(10).plot(kind='barh')
plt.title('Top 10 Important Features')
plt.xlabel('Importance Score')
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(8, 6))

# I'm using the 'y_pred' from the last fold of my validation loop.
# I plotted this histogram to see the shape of my model's predictions.
sns.histplot(y_pred, kde=True, color='green')

plt.title('Distribution of Predicted Delivery Times')
plt.xlabel('Predicted Time (minutes)')
plt.show()

# If this looks like a nice bell curve (Normal Distribution), it means the model is behaving stably.
# If it has weird spikes, the model might be biased towards specific numbers.

In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Step 1: Setup

# I'm using KFold again because we are still doing Regression.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

#Step 2: The Loop (Ensemble Strategy)

print("Starting Ensemble Training...")

for train_index, test_index in kf.split(X_scaled):

    # Splitting the data for this fold
    X_train, X_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Training Model 1: Random Forest
    # I'm keeping Random Forest as my base model.
    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    # Training Model 2: CatBoost
    # I set verbose=0 so it doesn't flood the screen with logs during training.
    cb_model = CatBoostRegressor(random_state=42, verbose=0)
    cb_model.fit(X_train, y_train)
    cb_pred = cb_model.predict(X_test)

    # Averaging Predictions (The Ensemble)
    # I'm taking the average of both models.
    # The idea is that if one model makes a mistake, the other might correct it.
    final_pred = (rf_pred + cb_pred) / 2

    # Calculating MAE on the averaged prediction
    mae = mean_absolute_error(y_test, final_pred)
    mae_scores.append(mae)

# Step 3: Final Results

average_mae = np.mean(mae_scores)

print(f"Ensemble MAE Scores per fold: {mae_scores}")
print(f"Average Ensemble MAE: {average_mae}")